<a href="https://colab.research.google.com/github/Phreely/Boltz-2_YAML_generator/blob/main/Boltz_2_yaml_advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Input protein sequence(s), then hit `Runtime` -> `Run all`
from google.colab import files
import os
import re
import hashlib
import requests
import yaml
import json
from string import ascii_uppercase

# User inputs
query_sequence = 'IASGGFRKYIAITGRRNVGKSSFMNALIGQEVSIVSNVAGTTTDPVFKSMELSPVGPITLIDTPGLDDVGELGIKRIKKAKKSLYRADCGILIVDDIPGNFEEQIIKLFKELEIPYFIAINKIDTIDHENIEKEYKKYNVPILKVSALKKIGFEKIGKTINSILPKDDEIPYLSDLIDGGDLVILVVPIDLGAPKGRLIMPQVHAIREGLDREALVLVVKERELRYAIENIGIKPRLVVTDSQSVMKVVSDVPEDIDLTTFSILESRYRGDLEYFVESVKAVENLKDGDTVIIMEGCTHRPLTEDIGRVKIPRWLTNHTGAALNLKVWAGVDMPELSEIEDAKLIIHCGGCVMNRNNMMRRVRMFKRLNIPMTNYGVVISYLHGVLERAIKPLMR:SNVPAELKYSKEHEWLRKEADGTYTVGITEHAQELLGDMVFVDLPEVGATVSAGDDCAVAESVKAASDIYAPVSGEIVAVNDALSDSPELVNSEPYAGGWIFKIKASDESELESLLDATAYEALLEDE'  #@param {type:"string"}
ligand_input_smiles = ''  #@param {type:"string"}
ligand_input_ccd = ''  #@param {type:"string"}
ligand_input_common_name = ''  #@param {type:"string"}
dna_input = ''  #@param {type:"string"}
jobname = 'HydF_Hmet'  #@param {type:"string"}

#@markdown ---
#@markdown ### **Advanced Settings**
#@markdown Enter a number for `max_msa`. Leave at 0 to use Boltz-2 defaults.
max_msa = 0 #@param {type:"integer"}
recycling_steps = "auto" #@param ["auto", "1", "3", "5", "10"]

#@markdown ---
#@markdown ### **Constraints Input**
#@markdown Enter constraints as a JSON array of objects.
constraints_input = "" #@param {type:"string"}
#@markdown **Constraint Types & Rules:**
#@markdown - **`contact`**: `[{"type": "contact", "atoms": ["A.10.CA", "B.25.CB"], "distance": 8.0, "force": false}]`
#@markdown - **`bond`**: `[{"type": "bond", "atoms": ["A.50.NZ", "LB.1.C1"]}]`
#@markdown - **`pocket`**: `[{"type": "pocket", "binder": "LB", "atoms": ["A.10.CA", "A.11.CA"], "distance": 5.0, "force": false}]`
#@markdown
#@markdown **Atom Formatting: `[ChainID].[ResidueNumber].[AtomName]`**
#@markdown - **Proteins**: `CA` is Alpha Carbon, `NZ` is Nitrogen in Lysine, etc. Residue number is its position in your sequence (1-indexed).
#@markdown - **Ligands**: Numbering can be tricky. Open the generated `.yaml` file to check the exact atom names and IDs assigned to your ligands, and then add your constraints!

#@markdown ---
#@markdown ### **Templates**
#@markdown Specify which protein chain should be modelled using a template and provide the template PDB ID.
template_chain = "" #@param {type:"string"}
template_pdb_id = "Insert link to PDB or CIF file here" #@param {type:"string"}

#@markdown ---
#@markdown ### **Affinity Prediction**
#@markdown Specify a ligand chain ID (e.g., `LB`, `CC`) to compute its binding affinity.
compute_affinity_ligand = "" #@param {type:"string"}

# 1. Clean up and Uppercase
query_sequence = re.sub(r'\s+', '', query_sequence).upper()
dna_input = re.sub(r'\s+', '', dna_input).upper()
ligand_input_smiles = re.sub(r'\s+', '', ligand_input_smiles) # SMILES are case-sensitive
ligand_input_ccd = re.sub(r'\s+', '', ligand_input_ccd).upper()
ligand_input_common_name = re.sub(r'\s+', '', ligand_input_common_name)

# 2. Setup Jobname and Directory
basejobname = re.sub(r'\W+', '', jobname)
jobname = basejobname + "_" + hashlib.sha1(query_sequence.encode()).hexdigest()[:5]
os.makedirs(jobname, exist_ok=True)

# 3. Handle Common Names via PubChem
def get_smiles(compound_name):
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{compound_name}/property/CanonicalSMILES/JSON"
        r = requests.get(url, timeout=5)
        return r.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
    except:
        return None

protein_sequences = query_sequence.split(':') if query_sequence else []
dna_sequences = dna_input.split(':') if dna_input else []
smiles_ligands = ligand_input_smiles.split(':') if ligand_input_smiles else []
ccd_ligands = ligand_input_ccd.split(':') if ligand_input_ccd else []

if ligand_input_common_name:
    for name in ligand_input_common_name.split(':'):
        smi = get_smiles(name)
        if smi:
            print(f"Found SMILES for {name}: {smi}")
            smiles_ligands.append(smi)

# 4. Construct YAML Dictionary
boltz_dict = {
    "name": jobname,
    "sequences": []
}

chain_gen = iter(ascii_uppercase)

# Add Proteins
for seq in protein_sequences:
    if seq:
        chain_id = next(chain_gen)
        prot_entry = {"protein": {"id": chain_id, "sequence": seq}}
        if template_chain and template_pdb_id and chain_id == template_chain.upper():
            prot_entry["protein"]["templates"] = [template_pdb_id]
        boltz_dict["sequences"].append(prot_entry)

# Add DNA
for seq in dna_sequences:
    if seq:
        boltz_dict["sequences"].append({"dna": {"id": next(chain_gen), "sequence": seq}})

# Add SMILES Ligands
for i, smi in enumerate(smiles_ligands):
    if smi:
        boltz_dict["sequences"].append({"ligand": {"id": f"L{next(chain_gen)}", "smiles": smi}})

# Add CCD Ligands
for ccd in ccd_ligands:
    if ccd:
        boltz_dict["sequences"].append({"ligand": {"id": f"C{next(chain_gen)}", "ccd": ccd}})

# 5. Advanced Parameters
sampling = {}
if recycling_steps != "auto":
    sampling["recycling_steps"] = int(recycling_steps)
if sampling:
    boltz_dict["sampling"] = sampling

if max_msa > 0:
    boltz_dict["msa"] = {"max_msa_seqs": max_msa}

# Parse and format constraints properly
def parse_atom_string(atom_str):
    parts = atom_str.split('.')
    if len(parts) >= 2 and parts[1].isdigit():
        parts[1] = int(parts[1])
    return parts

if constraints_input.strip():
    try:
        parsed_constraints = json.loads(constraints_input)
        formatted_constraints = []
        for c in parsed_constraints:
            ctype = c.get("type")
            if ctype == "contact":
                formatted_constraints.append({
                    "contact": {
                        "token1": parse_atom_string(c["atoms"][0]),
                        "token2": parse_atom_string(c["atoms"][1]),
                        "max_distance": c.get("distance", 6.0),
                        "force": c.get("force", False)
                    }
                })
            elif ctype == "bond":
                formatted_constraints.append({
                    "bond": {
                        "atom1": parse_atom_string(c["atoms"][0]),
                        "atom2": parse_atom_string(c["atoms"][1])
                    }
                })
            elif ctype == "pocket":
                formatted_constraints.append({
                    "pocket": {
                        "binder": c.get("binder"),
                        "contacts": [parse_atom_string(a) for a in c.get("atoms", [])],
                        "max_distance": c.get("distance", 6.0),
                        "force": c.get("force", False)
                    }
                })
        if formatted_constraints:
            boltz_dict["constraints"] = formatted_constraints
    except Exception as e:
        print(f"Warning: Could not parse constraints input correctly. Skipping constraints. Error: {e}")

# Add affinity properties if provided
if compute_affinity_ligand.strip():
    boltz_dict["properties"] = [{"affinity": {"binder": compute_affinity_ligand.strip()}}]

# 6. Save and Download
yaml_path = os.path.join(jobname, f"{jobname}.yaml")
with open(yaml_path, 'w') as f:
    yaml.dump(boltz_dict, f, default_flow_style=False, sort_keys=False)

print(f"\nSuccess! YAML created at {yaml_path}")
files.download(yaml_path)



Success! YAML created at HydF_Hmet_b0589/HydF_Hmet_b0589.yaml


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>